# Quick Re-train: EfficientNet-B0 Only\n\n**Run all cells in order.** Total time: ~70 min on T4 GPU.\n\n1. Setup + Install\n2. Download FER2013\n3. Prepare Data\n4. Define Model\n5. Train (2-phase)\n6. Save + Auto-Download


In [ ]:
# ================================================================
# CELL 1: ENVIRONMENT SETUP
# ================================================================
# Installs: PyTorch (CUDA), torchvision, efficientnet, grad-cam,
#           scikit-learn, matplotlib, seaborn
# Checks: GPU availability, CUDA version, memory
# Sets: Random seed for full reproducibility
# ================================================================

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q scikit-learn matplotlib seaborn tqdm kaggle
!pip install -q grad-cam

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    accuracy_score, precision_score, recall_score
)
from tqdm import tqdm
import copy
import os
import random
import math
import time
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 60)
print('ENVIRONMENT')
print('=' * 60)
print(f'PyTorch version:  {torch.__version__}')
print(f'CUDA available:   {torch.cuda.is_available()}')
if device.type == 'cuda':
    print(f'GPU:              {torch.cuda.get_device_name(0)}')
    total_gpu_mem = torch.cuda.mem_get_info()[1]
    print(f'GPU Memory:       {total_gpu_mem / 1e9:.1f} GB')
    print(f'CUDA version:     {torch.version.cuda}')
else:
    print('** WARNING: No GPU! Training will be extremely slow. **')
    print('** Go to Runtime -> Change runtime type -> GPU **')
print(f'Device:           {device}')
print('=' * 60)


In [ ]:
# ================================================================
# CELL 2: DOWNLOAD FER2013 DATASET
# ================================================================
# FER2013: 35,887 grayscale 48x48 face images across 7 emotions
# Published by Pierre-Luc Carrier and Aaron Courville (2013)
# Standard benchmark for facial emotion recognition
#
# To get kaggle.json:
#   1. Go to https://www.kaggle.com/settings
#   2. Click 'Create New Token' under API section
#   3. Upload the downloaded kaggle.json when prompted
# ================================================================

from google.colab import files

DATA_DIR = '/content/fer2013_data'

if not os.path.exists(f'{DATA_DIR}/train'):
    print('Upload your kaggle.json file:')
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    print('\nDownloading FER2013 dataset...')
    !kaggle datasets download -d msambare/fer2013 -p /content/
    !unzip -q /content/fer2013.zip -d {DATA_DIR}
    print('Download complete!')
else:
    print('Dataset already exists, skipping download.')

print('\nDataset structure:')
for split in ['train', 'test']:
    classes = sorted(os.listdir(f'{DATA_DIR}/{split}'))
    total = sum(len(os.listdir(f'{DATA_DIR}/{split}/{c}')) for c in classes)
    print(f'  {split}: {total:,} images across {len(classes)} classes')
    for c in classes:
        count = len(os.listdir(f'{DATA_DIR}/{split}/{c}'))
        print(f'    {c:10s}: {count:,}')


In [ ]:
# ================================================================
# CELL 3: DATASET EXPLORATION + VISUALIZATION
# ================================================================
# Analyzes class distribution, shows sample images,
# computes class weights for handling imbalance
# ================================================================

EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
NUM_CLASSES = 7

EMOTION_TO_CONFIDENCE = {
    'happy': 'Confident & Positive',
    'neutral': 'Calm & Composed',
    'surprise': 'Engaged & Attentive',
    'sad': 'Low Energy',
    'angry': 'Stressed / Tense',
    'fear': 'Nervous',
    'disgust': 'Uncomfortable'
}

CONFIDENCE_LEVEL = {
    'happy': 'HIGH', 'neutral': 'HIGH', 'surprise': 'MEDIUM',
    'sad': 'LOW', 'angry': 'LOW', 'fear': 'LOW', 'disgust': 'LOW'
}

# Count samples per class
train_counts = {e: len(os.listdir(f'{DATA_DIR}/train/{e}')) for e in EMOTIONS}
test_counts = {e: len(os.listdir(f'{DATA_DIR}/test/{e}')) for e in EMOTIONS}

# Class distribution plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['#e74c3c', '#9b59b6', '#f39c12', '#2ecc71', '#3498db', '#1abc9c', '#e67e22']
bars1 = axes[0].bar(train_counts.keys(), train_counts.values(), color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Training Set Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Images')
for bar, count in zip(bars1, train_counts.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{count:,}', ha='center', fontsize=9, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=45)

bars2 = axes[1].bar(test_counts.keys(), test_counts.values(), color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('Test Set Distribution', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Images')
for bar, count in zip(bars2, test_counts.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{count:,}', ha='center', fontsize=9, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# Imbalance ratio
max_count = max(train_counts.values())
min_count = min(train_counts.values())
print(f'\nClass imbalance ratio: {max_count/min_count:.1f}x (max/min)')
print(f'Largest class:  happy ({max_count:,} samples)')
print(f'Smallest class: disgust ({min_count:,} samples)')
print('This imbalance is why we need weighted loss function.\n')

# Compute class weights (inverse frequency)
total_train = sum(train_counts.values())
class_weights = []
print('Class weights for balanced training:')
for e in EMOTIONS:
    w = total_train / (NUM_CLASSES * train_counts[e])
    class_weights.append(w)
    print(f'  {e:10s}: {w:.4f}')
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

# Sample images (3 rows x 7 columns)
fig, axes = plt.subplots(3, 7, figsize=(18, 8))
for i, emotion in enumerate(EMOTIONS):
    folder = f'{DATA_DIR}/train/{emotion}'
    img_files = sorted(os.listdir(folder))[:3]
    for row in range(3):
        img = plt.imread(os.path.join(folder, img_files[row]))
        axes[row][i].imshow(img, cmap='gray')
        if row == 0:
            axes[row][i].set_title(f'{emotion.title()}\n({train_counts[emotion]:,})', fontsize=10)
        axes[row][i].axis('off')
fig.suptitle('Sample Images per Emotion Class (3 examples each)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ================================================================
# CELL 4: DATA PREPROCESSING + DATALOADERS
# ================================================================
# Key decisions:
#   - 224x224 resolution: standard for pretrained models, extracts
#     much richer features than 48x48
#   - Heavy augmentation on train: forces model to generalize
#     rather than memorize specific face positions
#   - No augmentation on val/test: fair evaluation
#   - 3-channel grayscale: repeat gray -> RGB for pretrained models
#   - ImageNet normalization: matches pretrained model statistics
#   - Stratified split preserves class proportions in train/val
# ================================================================

# ----- HYPERPARAMETERS -----
IMG_SIZE = 224
BATCH_SIZE = 32
VAL_SPLIT = 0.15
NUM_WORKERS = 2
# ---------------------------

# Training transforms: heavy augmentation to reduce overfitting
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

# Val/Test transforms: no augmentation, just resize + normalize
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_train_dataset = ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
full_train_eval = ImageFolder(f'{DATA_DIR}/train', transform=eval_transform)
test_dataset = ImageFolder(f'{DATA_DIR}/test', transform=eval_transform)

# Stratified-like split (random_split with fixed seed preserves rough proportions)
val_size = int(len(full_train_dataset) * VAL_SPLIT)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)
# Val uses eval transform (no augmentation)
_, val_dataset_eval = random_split(
    full_train_eval, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset_eval, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print('=' * 60)
print('DATA PIPELINE')
print('=' * 60)
print(f'Image size:        {IMG_SIZE}x{IMG_SIZE}x3 (upscaled from 48x48)')
print(f'Batch size:        {BATCH_SIZE}')
print(f'Train samples:     {len(train_dataset):,}')
print(f'Val samples:       {len(val_dataset):,}')
print(f'Test samples:      {len(test_dataset):,}')
print(f'Train batches:     {len(train_loader):,}')
print(f'Val batches:       {len(val_loader):,}')
print(f'Test batches:      {len(test_loader):,}')
print(f'Augmentations:     Flip, Rotation(15), Affine, ColorJitter, RandomErasing')
print(f'Normalization:     ImageNet (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])')

# Sanity check: visualize one batch
imgs, labels = next(iter(train_loader))
print(f'\nBatch tensor shape: {imgs.shape}')
print(f'Labels shape:       {labels.shape}')
print(f'Pixel range:        [{imgs.min():.3f}, {imgs.max():.3f}]')

# Show augmented samples
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
for i in range(16):
    ax = axes[i // 8][i % 8]
    img = imgs[i].permute(1, 2, 0).numpy()
    img = std * img + mean
    img = np.clip(img, 0, 1)
    ax.imshow(img[:, :, 0], cmap='gray')
    ax.set_title(EMOTIONS[labels[i]], fontsize=9)
    ax.axis('off')
fig.suptitle('Augmented Training Batch (what the model actually sees)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ================================================================
# CELL 5: MODEL DEFINITIONS (all 3 architectures)
# ================================================================

# ---- MODEL 1: Custom Deep CNN ----
class EmotionCNN(nn.Module):
    """6-block CNN with BatchNorm, Dropout, and progressive channel growth."""
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3 -> 32, 224 -> 112
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),

            # Block 2: 32 -> 64, 112 -> 56
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),

            # Block 3: 64 -> 128, 56 -> 28
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.15),

            # Block 4: 128 -> 256, 28 -> 14
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),

            # Block 5: 256 -> 512, 14 -> 7
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),

            # Block 6: 512 -> 512, 7 -> 3 (adaptive)
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((3, 3)),
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 3 * 3, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ---- MODEL 2: ResNet50 Transfer Learning ----
def create_resnet50(num_classes=7):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Sequential(
        nn.Linear(2048, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(512, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )
    return model


# ---- MODEL 3: EfficientNet-B0 Transfer Learning ----
def create_efficientnet(num_classes=7):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(1280, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )
    return model


# Instantiate all models
model_cnn = EmotionCNN(NUM_CLASSES).to(device)
model_resnet = create_resnet50(NUM_CLASSES).to(device)
model_effnet = create_efficientnet(NUM_CLASSES).to(device)

# Print parameter counts
print('=' * 70)
print('MODEL ARCHITECTURES')
print('=' * 70)
for name, model in [('Custom CNN', model_cnn), ('ResNet50', model_resnet), ('EfficientNet-B0', model_effnet)]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    print(f'  {name:20s} | Total: {total:>12,} | Trainable: {trainable:>10,} | Frozen: {frozen:>10,}')
print('=' * 70)


In [ ]:
# ================================================================
# CELL 6: TRAINING ENGINE
# ================================================================
# Advanced training with:
#   - Mixed Precision (AMP): uses float16 where safe, 2x speedup
#   - Cosine Annealing LR: smoothly decays LR following cosine curve
#   - Linear Warmup: gradually increases LR for first few epochs
#   - Weighted CrossEntropy: handles class imbalance
#   - Gradient Clipping: prevents exploding gradients
#   - Early Stopping: stops when val loss plateaus
#   - Best checkpoint saving by validation loss
# ================================================================

def get_cosine_schedule_with_warmup(optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
    """Cosine annealing with linear warmup."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return max(min_lr, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_model(model, train_loader, val_loader, epochs=50, lr=0.001,
                patience=8, model_name='Model', class_weights=None,
                warmup_epochs=3, use_amp=True):
    """Full training loop with AMP, cosine schedule, early stopping."""

    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=1e-4)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_epochs, epochs)
    scaler = GradScaler(enabled=use_amp)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'val_f1': [], 'lr': []
    }
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    start_time = time.time()

    print(f'\nTraining: {model_name}')
    print(f'  Epochs: {epochs} | LR: {lr} | Warmup: {warmup_epochs} | Patience: {patience}')
    print(f'  AMP: {use_amp} | Weight Decay: 1e-4 | Grad Clip: 1.0')
    print('=' * 90)
    print(f'{"Epoch":>6} | {"Train Loss":>10} {"Train Acc":>10} | '
          f'{"Val Loss":>10} {"Val Acc":>10} {"Val F1":>8} | {"LR":>10} | {"Status"}')
    print('-' * 90)

    for epoch in range(epochs):
        # ---- TRAIN ----
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()

            with autocast(enabled=use_amp):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix(loss=round(loss.item(), 4), acc=round(correct/total, 4))

        train_loss = running_loss / total
        train_acc = correct / total

        # ---- VALIDATE ----
        model.eval()
        val_loss_total = 0.0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                with autocast(enabled=use_amp):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                val_loss_total += loss.item() * imgs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss = val_loss_total / val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        current_lr = optimizer.param_groups[0]['lr']

        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr'].append(current_lr)

        # Checkpoint
        status = ''
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            status = 'BEST'
        else:
            patience_counter += 1
            status = f'wait {patience_counter}/{patience}'

        print(f'{epoch+1:>6} | {train_loss:>10.4f} {train_acc:>10.4f} | '
              f'{val_loss:>10.4f} {val_acc:>10.4f} {val_f1:>8.4f} | '
              f'{current_lr:>10.6f} | {status}')

        if patience_counter >= patience:
            print(f'\n  >>> Early stopping at epoch {epoch+1}')
            break

    elapsed = time.time() - start_time
    print('=' * 90)
    print(f'Training complete in {elapsed/60:.1f} minutes')
    print(f'Best val loss: {best_val_loss:.4f}')

    model.load_state_dict(best_model_state)
    return model, history


In [ ]:
# ================================================================
# CELL 9: TRAIN MODEL 3 — EfficientNet-B0 (Transfer Learning)
# ================================================================
# Same 2-phase strategy as ResNet50.
# EfficientNet is smaller but often more accurate due to NAS.
# Expected: ~64-71% accuracy. Takes ~10-15 min on T4.
# ================================================================

# Phase 1: Head only
print('EFFICIENTNET-B0 — PHASE 1: Classification head only')
model_effnet, history_effnet_p1 = train_model(
    model_effnet, train_loader, val_loader,
    epochs=15, lr=0.001, patience=5,
    model_name='EfficientNet-B0 Phase 1 (head)',
    class_weights=class_weights_tensor,
    warmup_epochs=2, use_amp=True
)

# Phase 2: Unfreeze and fine-tune
print('\nEFFICIENTNET-B0 — PHASE 2: Full fine-tuning')
for param in model_effnet.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model_effnet.parameters() if p.requires_grad)
print(f'  Unfrozen. Trainable params: {trainable:,}')

model_effnet, history_effnet_p2 = train_model(
    model_effnet, train_loader, val_loader,
    epochs=20, lr=0.00005, patience=6,
    model_name='EfficientNet-B0 Phase 2 (full)',
    class_weights=class_weights_tensor,
    warmup_epochs=2, use_amp=True
)

history_effnet = {k: history_effnet_p1[k] + history_effnet_p2[k] for k in history_effnet_p1}


In [ ]:
# ================================================================
# SAVE + AUTO-DOWNLOAD
# ================================================================
import os

save_dir = '/content/saved_models'
os.makedirs(save_dir, exist_ok=True)

# Evaluate on test set first
model_effnet.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Testing EfficientNet-B0'):
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast(enabled=True):
            outputs = model_effnet(imgs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
test_f1w = f1_score(all_labels, all_preds, average='weighted')
test_prec = precision_score(all_labels, all_preds, average='macro')
test_rec = recall_score(all_labels, all_preds, average='macro')

print(f'Test Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test Macro F1:  {test_f1:.4f}')
print(f'Test Weighted F1: {test_f1w:.4f}')
print(f'Test Precision: {test_prec:.4f}')
print(f'Test Recall:    {test_rec:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=EMOTIONS))

# Save model
fpath = f'{save_dir}/mockmate_efficientnet_b0.pt'
torch.save({
    'model_state_dict': model_effnet.state_dict(),
    'model_type': 'efficientnet_b0',
    'num_classes': NUM_CLASSES,
    'class_names': EMOTIONS,
    'emotion_to_confidence': EMOTION_TO_CONFIDENCE,
    'test_accuracy': test_acc,
    'test_f1_macro': test_f1,
    'test_f1_weighted': test_f1w,
    'test_precision': test_prec,
    'test_recall': test_rec,
    'img_size': IMG_SIZE,
    'normalization': {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},
}, fpath)

size_mb = os.path.getsize(fpath) / 1e6
print(f'\nSaved: {fpath} ({size_mb:.1f} MB)')

# AUTO-DOWNLOAD
from google.colab import files
print('\n>>> DOWNLOADING MODEL FILE NOW...')
files.download(fpath)
print('Check your browser downloads for mockmate_efficientnet_b0.pt!')
